# Create Multi-Agent Supervisor with Knowledge Assistant, Genie Space, and UC Functions

This notebook demonstrates how to create a Supervisor Agent using the AgentBricksManager class. The Supervisor Agent orchestrates multiple specialized agents:

**Components:**
- **Knowledge Assistant**: For analyzing financial documents (10-K/10-Q reports, earnings releases)
- **Genie Space**: For natural language SQL queries over ticker data
- **Unity Catalog Function**: For generating Vega-Lite chart specifications
- **You.com Web Search** (Optional): For real-time web search, content extraction, and research

**Prerequisites:**
- Run `01_instructor_setup_ka` to create the Knowledge Assistant
- Run `02_instructor_setup_genie` to create the Genie space
- Run `03_create_vegalite_uc_function_simple` to create the Vega-Lite UC Function
- (Optional) Run `03b_create_youdotcom_uc_functions` to create the You.com web search functions

**What this notebook does:**
- Discovers existing agents and functions
- Creates a Multi-Agent Supervisor that coordinates all tools
- Adds sample conversation starters
- Provides comprehensive financial analysis capabilities

In [0]:
# Import configuration and helper functions
import sys
import os
import logging
from databricks.sdk import WorkspaceClient

# Add parent directory to path to access resources
sys.path.append('..')
from resources.brick_setup_functions import AgentBricksManager

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger(__name__)

In [0]:
# Import configuration from config.py
exec(open('../config.py').read())

print("🚀 Starting Supervisor Agent creation...")
print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

In [0]:
# Initialize Workspace Client and AgentBricksManager
w = WorkspaceClient()
brick_manager = AgentBricksManager(w)

print("✅ Workspace Client and AgentBricksManager initialized")

In [ ]:
# Define the Supervisor Agent configuration
sa_name = "Supervisor_Agent_Mag7"
sa_description = "A multi-agent supervisor that coordinates financial analysis using Knowledge Assistant for document analysis, Genie for SQL queries, Unity Catalog functions for data visualization, and You.com for real-time web search."

sa_instructions = """You are a financial analysis supervisor agent that coordinates multiple specialized tools to provide comprehensive financial insights.

Your available tools:
1. Knowledge Assistant: Use for analyzing financial documents, earnings reports, 10-K/10-Q filings, and extracting insights
2. Genie Space: Use for querying ticker data with natural language SQL queries
3. Chart Generator (UC Function): Use for creating data visualizations in Vega-Lite format
4. Web Search (UC Function): Use for searching the internet for real-time information, news, and general knowledge
5. Web Content Extractor (UC Function): Use for reading and extracting full page content from URLs
6. Web Researcher (UC Function): Use for in-depth web research that returns synthesized, citation-backed answers

Workflow guidelines:
- For document-based questions, use the Knowledge Assistant
- For data queries about stock prices, volumes, or financial metrics, use the Genie space
- When users request charts or visualizations, use the Chart Generator to generate Vega-Lite specs
- For real-time information, current events, or general knowledge, use Web Search
- When you need to read a specific web page or article, use Web Content Extractor
- For comprehensive research on a topic, use Web Researcher
- Combine insights from multiple tools when providing comprehensive analysis
- Always cite sources and provide specific data points when available

Be helpful, accurate, and ensure all responses are well-sourced from the appropriate tools."""

print(f"Supervisor configuration defined:")
print(f"  Name: {sa_name}")
print(f"  Description: {sa_description[:80]}...")

In [0]:
# Discover Knowledge Assistant endpoint
ka_name = "Financial_Analysis_Assistant"
print(f"🔍 Looking for Knowledge Assistant: {ka_name}...")
ka_endpoint_name = None

try:
    ka_ids = brick_manager.find_by_name(ka_name)
    if ka_ids:
        ka_data = brick_manager.ka_get(ka_ids.tile_id)
        ka_endpoint_name = ka_data.get('knowledge_assistant', {}).get('endpoint_name')
        ka_status = ka_data.get('knowledge_assistant', {}).get('status', {}).get('endpoint_status')
        print(f"✅ Found KA: {ka_name} (tile_id: {ka_ids.tile_id})")
        print(f"   Endpoint: {ka_endpoint_name}, Status: {ka_status}")
    else:
        print(f"⚠️ KA '{ka_name}' not found. Make sure you've run 01_instructor_setup_ka first.")
except Exception as e:
    print(f"⚠️ Error looking up KA: {e}")

In [0]:
# Discover Genie space
genie_name = "Financial_Data_Explorer"
print(f"\n🔍 Looking for Genie space: {genie_name}...")
financial_genie_id = None

try:
    response = w.api_client.do('GET', '/api/2.0/genie/spaces')
    
    if 'spaces' in response:
        # Match by prefix to handle duplicates with suffixes like "(1)"
        for space in response['spaces']:
            space_title = space.get('title', '')
            if space_title.startswith(genie_name):
                financial_genie_id = space.get('space_id')
                print(f"✅ Found Genie space: {space_title} (ID: {financial_genie_id})")
                break
    
    if not financial_genie_id:
        print(f"⚠️ Genie space '{genie_name}' not found. Make sure you've run 02_instructor_setup_genie first.")
        
except Exception as e:
    print(f"⚠️ Error calling Genie spaces API: {e}")

In [0]:
# Verify UC Function
uc_function_name = "generate_vega_lite_spec"
full_uc_name = f"{catalog}.{schema}.{uc_function_name}"
print(f"\n🔍 Verifying UC Function: {full_uc_name}...")
try:
    func_info = w.functions.get(full_uc_name)
    print(f"✅ Found UC Function: {func_info.full_name}")
except Exception as e:
    print(f"⚠️ UC Function '{full_uc_name}' not found: {e}")

In [ ]:
# Verify You.com UC Functions (optional)
you_com_functions = ["you_web_search", "you_content_extract", "you_research"]
you_com_found = []

print(f"\n🔍 Checking for You.com UC Functions (optional)...")
for func_name in you_com_functions:
    full_name = f"{catalog}.{schema}.{func_name}"
    try:
        func_info = w.functions.get(full_name)
        you_com_found.append(func_name)
        print(f"✅ Found: {func_info.full_name}")
    except Exception:
        pass

if you_com_found:
    print(f"\n✅ Found {len(you_com_found)}/{len(you_com_functions)} You.com UC functions")
else:
    print(f"⚠️ No You.com UC functions found — web search will not be available.")
    print(f"   Run 03b_create_youdotcom_uc_functions to add web search capabilities.")

In [ ]:
# Check prerequisites
if not ka_endpoint_name:
    print("\n❌ Cannot create supervisor: KA endpoint not found.")
    print("Run 01_instructor_setup_ka first and wait for it to provision.")
    raise Exception("Knowledge Assistant endpoint not found")

# Create agents list - starting with KA and UC Function
agents = [
    {
        "name": "Financial_Documents_Assistant",
        "description": "Access to financial documents including 10-K/10-Q reports, earnings releases, and transcripts for comprehensive document analysis",
        "agent_type": "ka",
        "serving_endpoint": {"name": ka_endpoint_name}
    },
    {
        "name": "Chart_Generator",
        "description": "Generate Vega-Lite chart specifications for data visualization",
        "agent_type": "function",
        "unity_catalog_function": {
            "uc_path": {
                "catalog": catalog,
                "schema": schema,
                "name": "generate_vega_lite_spec"
            }
        }
    }
]

# Add Genie space agent if we found the ID
if financial_genie_id:
    genie_agent = {
        "name": "Ticker_Data_Explorer",
        "description": "Natural language SQL interface for querying financial ticker data including prices, volumes, and market metrics",
        "agent_type": "genie",
        "genie_space": {"id": financial_genie_id}
    }
    agents.append(genie_agent)
    print(f"✅ Added Genie space agent with ID: {financial_genie_id}")
else:
    print("⚠️ Genie space not found - creating MAS without Genie")

# Add You.com web search UC functions if they exist
you_com_agent_configs = {
    "you_web_search": {
        "name": "Web_Search",
        "description": "Search the web for real-time information, news, and general knowledge. Use for finding current events, company news, market data, and factual information from the internet.",
    },
    "you_content_extract": {
        "name": "Web_Content_Extractor",
        "description": "Extract and read full page content from a URL. Use when you need to read an article, blog post, or web page to answer a question or analyze its content.",
    },
    "you_research": {
        "name": "Web_Researcher",
        "description": "Perform deep web research on a topic. Returns synthesized, citation-backed answers. Use for in-depth analysis, comparisons, or when you need comprehensive, well-sourced answers.",
    }
}

if you_com_found:
    for func_name in you_com_found:
        config = you_com_agent_configs[func_name]
        agents.append({
            "name": config["name"],
            "description": config["description"],
            "agent_type": "function",
            "unity_catalog_function": {
                "uc_path": {
                    "catalog": catalog,
                    "schema": schema,
                    "name": func_name
                }
            }
        })
        print(f"✅ Added You.com UC function: {config['name']} -> {catalog}.{schema}.{func_name}")
else:
    print("ℹ️ Skipping You.com web search functions (not found)")

print(f"\n🤖 Creating Supervisor Agent...")
print(f"  Name: {sa_name}")
print(f"  Description: {sa_description[:80]}...")
print(f"\n🔧 Configured Agents ({len(agents)}):")
for i, agent in enumerate(agents):
    print(f"  {i+1}. {agent['name']} ({agent['agent_type']})")
    if 'serving_endpoint' in agent:
        print(f"      Endpoint: {agent['serving_endpoint']['name']}")
    elif 'genie_space' in agent:
        print(f"      Genie ID: {agent['genie_space']['id']}")
    elif 'unity_catalog_function' in agent:
        uc_path = agent['unity_catalog_function']['uc_path']
        print(f"      Function: {uc_path['catalog']}.{uc_path['schema']}.{uc_path['name']}")

In [0]:
# Create the Multi-Agent Supervisor
try:
    result = brick_manager.mas_create(
        name=sa_name,
        agents=agents,
        description=sa_description,
        instructions=sa_instructions
    )
    
    print("\n✅ Supervisor Agent created successfully!")
    
    # Extract key information from the result
    mas_info = result.get('multi_agent_supervisor', {})
    tile_info = mas_info.get('tile', {})
    sa_id = tile_info.get('tile_id')
    sa_name_created = tile_info.get('name', sa_name)
    
    print(f"  Multi-Agent Supervisor ID: {sa_id}")
    print(f"  Name: {sa_name_created}")
    print(f"  Description: {mas_info.get('description', 'No description')}")
    
    # Display configured agents
    configured_agents = mas_info.get('agents', [])
    print(f"\n🔧 Configured Agents ({len(configured_agents)}):")
    for i, agent in enumerate(configured_agents):
        agent_name = agent.get('name', 'Unknown')
        agent_type = agent.get('agent_type', 'Unknown')
        print(f"  {i+1}. {agent_name} ({agent_type})")
        
except Exception as e:
    print(f"❌ Error creating Multi-Agent Supervisor: {e}")
    raise

In [ ]:
# Check the status of the Multi-Agent Supervisor
print(f"\n🔍 Checking Multi-Agent Supervisor status...")
try:
    status = brick_manager.mas_get_endpoint_status(sa_id)
    print(f"Status: {status}")
    
    if status in ['PROVISIONING', 'NOT_READY']:
        print("Multi-Agent Supervisor is still provisioning. You can check its status in the Databricks workspace.")
    elif status == 'ONLINE':
        print("✅ Multi-Agent Supervisor is ready to use!")
    else:
        print(f"Multi-Agent Supervisor status: {status}")
        
except Exception as e:
    print(f"⚠️ Warning: Could not check status: {e}")

# Define sample conversation starters
sample_conversations = [
    {
        "query": "What were Apple's key revenue highlights in their latest quarterly report?",
        "expected_flow": "Knowledge Assistant → Document analysis"
    },
    {
        "query": "Show me a chart of Tesla's stock price trends over the last week",
        "expected_flow": "Genie → Data query → UC Function → Visualization"
    },
    {
        "query": "Compare NVIDIA's recent earnings performance with their guidance and create a visualization",
        "expected_flow": "Knowledge Assistant → Genie → UC Function → Comprehensive analysis"
    },
    {
        "query": "What are the main risk factors mentioned in Meta's 10-K filing?",
        "expected_flow": "Knowledge Assistant → Document analysis"
    },
    {
        "query": "Create a bar chart showing the average trading volume for each company in our dataset",
        "expected_flow": "Genie → Data query → UC Function → Visualization"
    },
    {
        "query": "Search the web for the latest news on NVIDIA AI chip competition",
        "expected_flow": "Web Search → Real-time web results"
    },
    {
        "query": "Research how Databricks compares to Snowflake in 2025 and summarize the key differences",
        "expected_flow": "Web Researcher → Synthesized research with citations"
    }
]

print(f"\n📚 Sample Conversation Starters ({len(sample_conversations)}):")
for i, example in enumerate(sample_conversations):
    print(f"  {i+1}. \"{example['query']}\"")
    print(f"      Expected flow: {example['expected_flow']}")

## 🎉 Multi-Agent Supervisor Setup Complete!

### 📋 Summary

Your Multi-Agent Supervisor has been configured with:
- **Knowledge Assistant**: Financial document analysis (10-K, 10-Q, earnings)
- **Genie Space**: Natural language SQL queries over ticker data
- **Unity Catalog Function**: Vega-Lite chart generation
- **You.com Web Search** (if available): Real-time web search, content extraction, and research

### 🚀 Next Steps

1. **Wait for provisioning** if the status is still PROVISIONING
2. **Access the supervisor** in Databricks workspace
   - Navigate to **Machine Learning > Agents**
   - Look for: **Supervisor_Agent_Mag7**
3. **Test multi-agent coordination** with sample queries
4. **Experiment with complex queries** that require multiple agents

### 💡 Example Multi-Tool Queries

These queries demonstrate how the supervisor coordinates multiple agents:

1. **Document + Data Analysis**:
   - "Compare NVIDIA's recent earnings performance with their guidance and create a visualization"
   
2. **Data Query + Visualization**:
   - "Show me a chart of Tesla's stock price trends over the last week"
   
3. **Web Search + Analysis**:
   - "Search the web for the latest NVIDIA AI chip news and compare with our earnings data"

4. **Deep Research**:
   - "Research how Databricks compares to Snowflake in 2025"

### 🏛️ Architecture Overview

```
📄 Knowledge Assistant  ← Financial documents (10-K, 10-Q, earnings)
📊 Genie Space          ← Ticker data (ticker_data table)
📈 Chart Generator      ← Vega-Lite specs (generate_vega_lite_spec)
🔍 Web Search           ← You.com free tier (you_web_search)
📖 Content Extractor    ← You.com free tier (you_content_extract)
🔬 Web Researcher       ← You.com free tier (you_research)
🤖 Multi-Agent Supervisor ← Coordinates all agents
```

### ⚠️ Preview Feature Requirement

Note: Creating Multi-Agent Supervisors requires the **"Agent Framework: On-Behalf-Of-User Authorization"** preview feature to be enabled in your workspace.

In [ ]:
# Summary
print("\n🎉 Multi-Agent Supervisor Setup Complete!")
print("\n📋 Summary:")
print(f"  • Name: {sa_name}")
print(f"  • Multi-Agent Supervisor ID: {sa_id}")
print(f"  • Agents Configured: {len(agents)}")
print(f"    - Knowledge Assistant (Financial Documents)")
if financial_genie_id:
    print(f"    - Genie Space (Ticker Data)")
print(f"    - Unity Catalog Function (Visualization)")
if you_com_found:
    for func_name in you_com_found:
        print(f"    - You.com: {you_com_agent_configs[func_name]['name']}")

print("\n🚀 Next Steps:")
print("1. Wait for the Multi-Agent Supervisor to be ONLINE if it's still provisioning")
print("2. Test the supervisor in the Databricks workspace (Machine Learning > Agents)")
print("3. Try the sample conversation starters to test multi-agent coordination")
print("4. Experiment with complex queries that require multiple agents")

print(f"\n🔗 Access your Multi-Agent Supervisor:")
print(f"   Navigate to Machine Learning > Agents in your Databricks workspace")
print(f"   Look for: {sa_name}")

print(f"\n💡 Example multi-tool queries to try:")
for example in sample_conversations[:3]:
    print(f"   • \"{example['query']}\"")

print(f"\n🏛️ Architecture Overview:")
print(f"   📄 Knowledge Assistant  ← Financial documents (10-K, 10-Q, earnings)")
if financial_genie_id:
    print(f"   📊 Genie Space          ← Ticker data ({catalog}.{schema}.ticker_data)")
print(f"   📈 Chart Generator      ← {catalog}.{schema}.generate_vega_lite_spec")
if you_com_found:
    print(f"   🔍 Web Search           ← You.com free tier (no API key)")
    print(f"   📖 Content Extractor    ← You.com free tier (no API key)")
    print(f"   🔬 Web Researcher       ← You.com free tier (no API key)")
print(f"   🤖 Multi-Agent Supervisor ← Coordinates all agents")